# GlioStrata — Data Exploration

Quick exploration of the synthetic GlioStrata cohort. Once real TCGA/CGGA data
are ingested (see `docs/roadmap.md`), the same notebook structure applies.

**Prerequisites**: `pip install -e ".[dev,heavy]"` (heavy for matplotlib/seaborn).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gliostrata.data.ingest import generate_synthetic_cohort
from gliostrata.features.store import build_feature_store
from gliostrata.optimization.bayesian import optimize_configurations, evaluate_personalization
from gliostrata.prediction.survival import predict_survival
from gliostrata.profiling.states import deconvolve_states
from gliostrata.profiling.subtype import classify_subtypes
from gliostrata.profiling.tme import deconvolve_tme

# Matplotlib Chinese-safe defaults (kept for consistency with project conventions)
import matplotlib.font_manager as fm
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. Generate a synthetic cohort

In [ ]:
cohort = generate_synthetic_cohort(n=200, seed=42)
print(f'Cohort shape: {cohort.shape}')
cohort.head()

## 2. Subtype distribution + survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cohort['subtype'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Verhaak subtype distribution')
axes[0].set_ylabel('Count')

cohort.boxplot(column='overall_survival_months', by='subtype', ax=axes[1])
axes[1].set_title('Overall survival by subtype')
axes[1].set_ylabel('Months')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 3. Build the feature store + predict survival

In [ ]:
from gliostrata.data.harmonize import harmonize
harmonized = harmonize(cohort)
fs = build_feature_store(harmonized, classify_subtypes(harmonized), deconvolve_states(harmonized), deconvolve_tme(harmonized))
surv = predict_survival(fs, seed=42)
print(f'Feature store: {fs.shape}')
print(f'Median predicted OS: {surv["predicted_survival_months"].median():.1f} months')

## 4. Run the optimizer + check personalization

In [ ]:
configs = optimize_configurations(fs, surv, n_top=5, n_candidates=100, seed=42)
pers = evaluate_personalization(configs)
print(f'Personalization: {pers["n_distinct_top5_cell_payload_combos"]} distinct top-5 cell×payload combos')
print(f'Threshold ≥ {pers["threshold_min_distinct"]}: {"PASS" if pers["passes_threshold"] else "FAIL"}')

# Survival gain distribution (T5 stub on synthetic data)
top1 = configs[configs['rank'] == 1]
print(f'\nT5 stub: {(top1["survival_gain_vs_stupp_months"] >= 3).mean():.0%} with ≥3-month gain')
print(f'Harm fraction: {(top1["survival_gain_vs_stupp_months"] < 0).mean():.0%}')

## 5. Next steps

- Replace synthetic data with real TCGA ingestion (L0 stub → real).
- Train real subtype classifier (L2 stub → ensemble on Verhaak signatures).
- Swap random-search optimizer for BoTorch Bayesian optimization (L6).

See `docs/roadmap.md` for the recommended implementation order.